Here's your plan:

- input → tokens
- token embedding + positional embedding (add them)
- one-head attention ✓ (done)
- multi-head attention — heads in parallel, concat, + an output projection back to n_embd
- feedforward (Linear → ReLU → Linear, 4× wide) ← was missing
- assemble a Block = LN→MHA→+residual, LN→FFN→+residual
- stack N blocks (depth)
- final LayerNorm + lm_head (Linear n_embd → vocab_size to get logits) ← easy to forget
- forward → cross-entropy loss (reshape logits to (B*T, vocab), targets to (B*T,))
- backward + optimizer step (zero_grad → backward → step, AdamW) — this is your "training loop"
- sampling / generate

In [80]:
import torch 
import torch.nn.functional as F
import torch.nn as nn 

In [81]:
with open('./input.txt','r') as f:
    x = f.read()

In [82]:
T = 4
B = 10
C = 4
heads = 2

In [83]:
inp = torch.randint(1,len(x)-T,(B,))
inp = [x[i.item():i.item()+T] for i in inp]
stoi = {}
itos = {}
for i,s in enumerate(sorted(list(set(x)))):
    stoi[s] = i 
    itos[i] = s 


inp2 = []
for i in inp:
    inp2.append([stoi[j] for j in i])

inp2 = torch.tensor(inp2)
inp2

tensor([[21,  1, 46, 39],
        [31, 32, 13, 26],
        [56,  1, 21,  1],
        [10,  0, 14, 63],
        [51,  1, 63, 53],
        [ 1, 51, 47, 52],
        [42, 43, 39, 56],
        [46, 47, 57,  1],
        [46, 39, 58,  1],
        [ 1, 58, 46, 47]])

cleaner implementaion 

In [101]:
class block(nn.Module):
    def __init__(self):
        super().__init__()
        assert C%heads == 0 , "C must be divisible by heads"
        self.head_size = C//heads
        self.key = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.query = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.value = nn.ModuleList([nn.Linear(C,self.head_size,bias=False) for _ in range(heads)])
        self.proj = nn.Linear(C,C)
        self.feedfor = nn.Sequential(nn.Linear(C,4*C), nn.ReLU(), nn.Linear(4*C,C))
        self.ln1 = nn.LayerNorm(C)
        self.ln2 = nn.LayerNorm(C)
        



    def attention(self,x):
        x = self.ln1(x)
        out = None
        for i in range(heads):
            query = self.query[i](x)
            key = self.key[i](x)
            value = self.value[i](x)
            
            qk = query@key.transpose(-2,-1) * self.head_size**-0.5
            tril = torch.tril(torch.ones(T,T))
            we = qk.masked_fill(tril == 0, float('-inf'))
            we = torch.softmax(we,dim=-1)
            temp = we@value
            if out is None:
                out = temp
            else:
                out = torch.cat([out,temp],dim=-1)
        return self.proj(out)

    def feedf(self,x):
        x = self.ln2(x)
        return self.feedfor(x)
        
    def forward(self,x):
        att = x+self.attention(x)
        fow = att+self.feedf(att)
        return fow


In [105]:
class GPT(nn.Module):

    def __init__(self,no_of_block):
        super().__init__()
        self.embd = nn.Embedding(len(stoi),C)
        self.posi = nn.Embedding(T,C)
        self.block1 = nn.Sequential(*[block() for _ in range(no_of_block)])


    def nn_embed(self,x):
        embedings = self.embd(x)
        embedings = embedings+self.posi(torch.arange(T))
        return embedings

    def block(self,e):
        return self.block1(e)


In [106]:
gpt = GPT(3)
emb = gpt.nn_embed(inp2)
rblock = gpt.block(emb)


In [107]:
rblock

tensor([[[ 0.3826, -0.6545,  0.2583, -0.7907],
         [ 0.3721,  4.0472,  2.3988,  0.7371],
         [ 2.7494,  0.7442,  2.0693, -2.0256],
         [-1.7787, -1.9673,  1.9342,  0.3408]],

        [[-0.8602,  0.9052,  1.1404, -1.9099],
         [-1.3333,  3.9869,  0.3384,  1.2068],
         [ 0.9364,  0.2069,  1.1380, -0.2463],
         [-1.0689, -0.5754,  1.5857, -0.6146]],

        [[ 0.2626,  0.5160,  0.3621, -2.6717],
         [ 0.3993,  4.2148,  2.3334,  0.6426],
         [ 0.4379, -0.4163,  0.8157,  0.0579],
         [-0.2197,  1.1939,  1.7502,  1.2090]],

        [[ 0.9647,  2.2660, -0.1579, -2.9571],
         [-0.4478,  2.4012,  2.0828, -1.0196],
         [-0.3143,  1.0379,  0.3793, -2.9194],
         [ 0.5045, -0.3714, -1.7291, -0.7511]],

        [[ 0.2325,  1.7919, -2.0031,  0.0733],
         [ 0.8591,  3.7731,  1.8160,  0.4430],
         [ 1.4353,  1.2598, -1.0976, -1.3544],
         [-0.2561, -3.6017,  1.4193, -0.8044]],

        [[ 0.8466,  2.7680,  1.1948, -0.9058],
   

In [97]:
for name, p in gpt.named_parameters():
    print(name, tuple(p.shape))


key.0.weight (2, 4)
key.1.weight (2, 4)
query.0.weight (2, 4)
query.1.weight (2, 4)
value.0.weight (2, 4)
value.1.weight (2, 4)
proj.weight (4, 4)
proj.bias (4,)
feedfor.0.weight (16, 4)
feedfor.0.bias (16,)
feedfor.2.weight (4, 16)
feedfor.2.bias (4,)
ln1.weight (4,)
ln1.bias (4,)
ln2.weight (4,)
ln2.bias (4,)
embd.weight (65, 4)
posi.weight (4, 4)
block1.0.key.0.weight (2, 4)
block1.0.key.1.weight (2, 4)
block1.0.query.0.weight (2, 4)
block1.0.query.1.weight (2, 4)
block1.0.value.0.weight (2, 4)
block1.0.value.1.weight (2, 4)
block1.0.proj.weight (4, 4)
block1.0.proj.bias (4,)
block1.0.feedfor.0.weight (16, 4)
block1.0.feedfor.0.bias (16,)
block1.0.feedfor.2.weight (4, 16)
block1.0.feedfor.2.bias (4,)
block1.0.ln1.weight (4,)
block1.0.ln1.bias (4,)
block1.0.ln2.weight (4,)
block1.0.ln2.bias (4,)
block1.1.key.0.weight (2, 4)
block1.1.key.1.weight (2, 4)
block1.1.query.0.weight (2, 4)
block1.1.query.1.weight (2, 4)
block1.1.value.0.weight (2, 4)
block1.1.value.1.weight (2, 4)
block1.1.